# Importing Libraries

In [ ]:
import pandas as pd
import os
import torch
import transformers
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder


print(f"Using transformers version: {transformers.__version__}")
print(f"Using tensorflow version: {tf.__version__}")
print(f"Using torch version: {torch.__version__}")

Using transformers version: 5.12.1
Using tensorflow version: 2.20.0
Using torch version: 2.11.0+cu128


In [ ]:
!pip install vaderSentiment

In [ ]:
from huggingface_hub import login

login()

# Data Preprocessing


## Dataset Loading

### Dataset 1: Sentiment Analysis for Mental Health

Source: https://www.kaggle.com/datasets/suchintikasarkar/sentiment-analysis-for-mental-health

In [ ]:
df = pd.read_csv("/content/risk_analysis.csv")
df.head()

,Unnamed: 0,statement,status
0,0,oh my gosh,Mid-Risk
1,1,"trouble sleeping, confused mind, restless hear...",Mid-Risk
2,2,"All wrong, back off dear, forward doubt. Stay ...",Mid-Risk
3,3,I've shifted my focus to something else but I'...,Mid-Risk
4,4,"I'm restless and restless, it's been a month n...",Mid-Risk


In [ ]:
df['status'].value_counts()

,count
status,
Mid-Risk,19487
Low-Risk,16343
High-Risk,12064


### Dataset 2: Suicidal Tweet Detection Dataset

Source: https://www.kaggle.com/datasets/aunanya875/suicidal-tweet-detection-dataset

In [ ]:
df2=pd.read_csv("/content/Suicide_Ideation_Dataset(Twitter-based).csv")
df2.head()

,Tweet,Suicide
0,making some lunch,Not Suicide post
1,@Alexia You want his money.,Not Suicide post
2,@dizzyhrvy that crap took me forever to put to...,Potential Suicide post
3,@jnaylor #kiwitweets Hey Jer! Since when did y...,Not Suicide post
4,Trying out &quot;Delicious Library 2&quot; wit...,Not Suicide post


In [ ]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1787 entries, 0 to 1786
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   Tweet    1785 non-null   object
 1   Suicide  1787 non-null   object
dtypes: object(2)
memory usage: 28.1+ KB


In [ ]:
df2['Suicide'].value_counts()

,count
Suicide,
Not Suicide post,1127
Potential Suicide post,660


## Merge and Prepare the Dataset

In [ ]:
high_risk_tweets = df2[df2['Suicide'] == 'Potential Suicide post '].copy()
low_risk_tweets = df2[df2['Suicide'] == 'Not Suicide post'].copy()

new_high_risk_df=pd.DataFrame({
    'statement': high_risk_tweets['Tweet'],
    'status': "High-Risk"
})

new_low_risk_df=pd.DataFrame({
    'statement': low_risk_tweets['Tweet'],
    'status': "Low-Risk"
})

df=pd.concat([df, new_high_risk_df], ignore_index=True)
df=pd.concat([df, new_low_risk_df], ignore_index=True)

print("New 'status' distribution:")
display(df['status'].value_counts())

New 'status' distribution:


,count
status,
Mid-Risk,19487
Low-Risk,17470
High-Risk,12724


## Encoding the Risk Labels

0 - High Risk

1 - Low Risk

2 - Moderate Risk

In [ ]:
le = LabelEncoder()
df['status'] = le.fit_transform(df['status'])
df['status'].value_counts()

,count
status,
2,19487
1,16343
0,12064


## Handling Missing Data

In [ ]:
df.drop(columns=["Unnamed: 0"], inplace=True)
df.dropna(inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49679 entries, 0 to 49680
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   statement        49679 non-null  object 
 1   status           49679 non-null  int64  
 2   sentiment_score  49679 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.5+ MB


## Splitting Dataset into Train and Test Dataset

In [ ]:
train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["statement"].to_numpy(), df["status"].to_numpy(), test_size=0.2, random_state=42, stratify=df["status"].to_numpy()
)
len(train_texts), len(test_texts), len(train_labels), len(test_labels)

(38315, 9579, 38315, 9579)

## Uploading Dataset into HuggingFace Hub

In [ ]:
from huggingface_hub import create_repo, upload_file

repo_id = "gokulan006/risk-analysis-dataset"

create_repo(
    repo_id=repo_id,
    repo_type="dataset",
    exist_ok=True
)

upload_file(
    path_or_fileobj="/content/Reddit_risk_analysis.csv",
    path_in_repo="Reddit_risk_analysis.csv",
    repo_id=repo_id,
    repo_type="dataset"
)

# Sentiment Score Analyzing Across Risk Categories

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

df["sentiment_score"] = df["statement"].apply(
    lambda x: analyzer.polarity_scores(str(x))["compound"]
)

In [ ]:
df.groupby('status')['sentiment_score'].mean(),df.groupby('status')['sentiment_score'].min(),df.groupby('status')['sentiment_score'].max()

(status
 0   -0.338546
 1    0.112801
 2   -0.320461
 Name: sentiment_score, dtype: float64,
 status
 0   -0.9999
 1   -0.9895
 2   -0.9997
 Name: sentiment_score, dtype: float64,
 status
 0    0.9997
 1    0.9981
 2    0.9996
 Name: sentiment_score, dtype: float64)

# DistilBERT Model Development

## Loading the Pre-Trained DistilBERT Model and Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


## Tokenizing the Dataset

In [ ]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)


test_encodings = tokenizer(
    test_texts.tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

## Creating PyTorch Dataset

In [ ]:
import torch
from torch.utils.data import Dataset

class RiskDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(int(self.labels[idx]))
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
import sklearn.preprocessing as preprocessing


train_dataset = RiskDataset(
    train_encodings,
    train_labels.tolist()
)

test_dataset = RiskDataset(
    test_encodings,
    test_labels.tolist()
)

## Configure Traning Parameters

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./distillbert_model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True
)

import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

## Training the Model

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.265688,0.428217,0.854473,0.853203
2,0.236654,0.504263,0.858023,0.857080
3,0.154157,0.689014,0.857918,0.856937


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=14370, training_loss=0.2422676275236041, metrics={'train_runtime': 1466.5384, 'train_samples_per_second': 78.378, 'train_steps_per_second': 9.799, 'total_flos': 3806684170225920.0, 'train_loss': 0.2422676275236041, 'epoch': 3.0})

## Evaluating the Model

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,F1
0.154157,0.428217,3,0.854473,0.853203


{'eval_loss': 0.4282165467739105,
 'eval_accuracy': 0.8544733270696315,
 'eval_f1': 0.8532027955317172}

## Saving the Model

In [ ]:
trainer.save_model("./distillbert_model")
tokenizer.save_pretrained("./distillbert_model")

## Uploading the Model into HuggingFace Hub

In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="./distillbert_model",
    repo_id="gokulan006/distilbert-reddit-mental-health-risk-classifier",
    repo_type="model"
)

CommitInfo(commit_url='https://huggingface.co/gokulan006/distilbert-reddit-mental-health-risk-classifier/commit/66462057b65dd545e86aa4975e43c0144459c150', commit_message='Upload folder using huggingface_hub', commit_description='', oid='66462057b65dd545e86aa4975e43c0144459c150', pr_url=None, repo_url=RepoUrl('https://huggingface.co/gokulan006/distilbert-reddit-mental-health-risk-classifier', endpoint='https://huggingface.co', repo_type='model', repo_id='gokulan006/distilbert-reddit-mental-health-risk-classifier'), pr_revision=None, pr_num=None)

## Loading the Model from HuggingFace Hub

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = "gokulan006/distilbert-reddit-mental-health-risk-classifier"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

## Testing the Model

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

risk_labels = [
    "High Risk",
    "Low Risk",
    "Moderate Risk"
]

def predict_risk(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim=1
    )

    predicted_class = torch.argmax(
        probabilities,
        dim=1
    ).item()

    confidence = probabilities[0][predicted_class].item()

    return {
        "risk": risk_labels[predicted_class],
        "confidence": round(confidence * 100, 2)
    }

In [3]:
texts=["I don't see any reason to keep living anymore. Everyone would be better off without me",
       "I've been feeling depressed and anxious for weeks. I can't focus on anything.",
       "I had a great time with my friends this weekend and we watched a movie."]

for text in texts:
    print(f"Text:{text} and Prediction {predict_risk(text)}")

Text:I don't see any reason to keep living anymore. Everyone would be better off without me and Prediction {'risk': 'High Risk', 'confidence': 52.86}
Text:I've been feeling depressed and anxious for weeks. I can't focus on anything. and Prediction {'risk': 'Moderate Risk', 'confidence': 99.57}
Text:I had a great time with my friends this weekend and we watched a movie. and Prediction {'risk': 'Low Risk', 'confidence': 99.87}
